# AUC precision-recall plots for different detectors



In [16]:
import os
import sys


sys.path.append(os.path.abspath(".."))
from pathlib import Path
from genai_detection.config import CONFIG

path2results = (
    Path(os.getcwd()).resolve().parent
    / CONFIG.SAVE_PATH
    / "impostor_scores"
    / "our_contributions"
    / "auc_recall_prec"
)
if not path2results.exists():
    path2results.mkdir(parents=True, exist_ok=True)

path2dir = (
    Path(os.getcwd()).resolve().parent
    / CONFIG.SAVE_PATH
    / "impostor_scores/our_contributions/precision_recall_values"
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import auc

auc_pr_per_appr = {}

# Load the CSV file
# CSV format:
# threshold,precision,recall
for file in path2dir.glob("*.csv"):
    if "different" in file.name:
        continue
    # print(f"Processing file: {file}")
    appr_name = file.stem.split("_")[0]
    if appr_name == "unsupervised":
        appr_name = "_".join(file.stem.split("_")[:3])
    # print("Approach:", appr_name)
    data = pd.read_csv(file)

    # Extract recall and precision
    recall = data["recall"].values
    precision = data["precision"].values

    # Sort by recall (important for correct AUC calculation)
    sorted_indices = np.argsort(recall)
    recall = recall[sorted_indices]
    precision = precision[sorted_indices]

    # Calculate AUC using trapezoidal rule
    auc_pr = auc(recall, precision)
    auc_pr_per_appr[appr_name] = np.round(auc_pr, 2)
    # print(f"Area under Precision-Recall curve (AUC-PR) for {appr_name}:", auc_pr)

translation_dict = {
    "naive": "One-Step Paraphraser (LLM)",
    "on-the-fly": "On-the-Fly",
    "unsupervised_baseline_min-max": "Unsup. Min-Max (B)",
    "unsupervised_baseline_cosine": "Unsup. Cosine (B)",
    "supervised": "Sup. SVM (B)",
    "unmasking": "Unmasking",
    "ppmd": "PPMd",
    "fixed": "Fixed",
}
auc_pr_per_appr = {translation_dict.get(k, k): v for k, v in auc_pr_per_appr.items()}
auc_pr_per_appr = pd.DataFrame.from_dict(
    auc_pr_per_appr, orient="index", columns=["AUC-PR"]
)
auc_pr_per_appr = auc_pr_per_appr.sort_values(by="AUC-PR", ascending=False)
print(auc_pr_per_appr)
auc_pr_per_appr.to_csv(path2results / "auc_pr_all_detectors.csv")

                            AUC-PR
PPMd                          0.69
Fixed                         0.68
Unsup. Min-Max (B)            0.68
One-Step Paraphraser (LLM)    0.64
Unsup. Cosine (B)             0.52
Unmasking                     0.51
Sup. SVM (B)                  0.49
